In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..', '..', '..'))
sys.path.insert(0, module_path)

### Подготовка
1. Загрузим данные
2. Выкинем implicit feedback в виде нулей
3. Time-split на train и test. 90% данных оставим на train, 10% на тест

In [2]:

from brs.models.rate_prediction.baseline.model import BaselineRateModel
from brs.ingestion import LoadLocalDataset

lld = LoadLocalDataset()
data_books = lld.load_dataset("books", clear_version=True)
data_users = lld.load_dataset("users", clear_version=True)
data_ratings = lld.load_dataset("ratings", clear_version=True)

In [3]:
data_books.dtypes

ISBN                    object
Book-Title              object
Book-Author             object
Year-Of-Publication    float64
Publisher               object
dtype: object

In [4]:
from brs.models.rate_prediction.lin_reg.pipeline import RatingsModelPipeline, UsersModelPipeline
data_ratings = RatingsModelPipeline().fit_transform(data_ratings)
data_users = UsersModelPipeline(ratings_df=data_ratings).fit_transform(data_users)

In [5]:
num_ratings = data_ratings.shape[0]
split = round(num_ratings * 0.9)
train, test = data_ratings[:split], data_ratings[split:]

### Pipeline/preprocess

- Считаем среднюю оценку для каждой книги на train
- Предсказываем каждому пользователю на test именно её

In [ ]:
from brs.preprocessing.pipeline import NaPreprocessing, DummiesPreprocessing, TopKDummiesPreprocessing, DropPreprocessing
from brs.pipelines.dataframe import DataframePipeline
from brs.feature_engineering.pipeline import UserSetAgeFeature, UserSetCountryFeature, UserSetNumRatingFeature, UserSetMeanRatingFeature

Na_dict = {
    "Age": {"strategy": "mean"},
    "Country": {"strategy": "most_frequent"}, 
}

users_pipeline = DataframePipeline([
    UserSetCountryFeature(),
    NaPreprocessing({
        "Age": {"strategy": "mean"},
        "Country": {"strategy": "most_frequent"}, 
    }),
    UserSetAgeFeature(),
    UserSetNumRatingFeature(ratings_df=train),
    UserSetMeanRatingFeature(ratings_df=train),
    NaPreprocessing({
        "Num-Ratings": {"strategy": "mean"},
        "Mean-Rating": {"strategy": "mean"}
    }),
    TopKDummiesPreprocessing(columns=["Country"], top_k=15, drop_first=True),
    DummiesPreprocessing(columns=["Age-Group"]),
    DropPreprocessing(columns=["Location"])
])

data_users = users_pipeline.fit_transform(data_users)

data_users.head()

,User-ID,Location,Age,Num-Ratings,Mean-Rating,austria,canada,finland,france,germany,...,switzerland,united kingdom,usa,Other,Age-Group_0-12,Age-Group_13-17,Age-Group_18-29,Age-Group_30-44,Age-Group_45-59,Age-Group_60+
0,8,"timmins, ontario, canada",35.875148,7.0,5.571429,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,99,"franktown, colorado, usa",42.000000,8.0,7.625000,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
2,160,"almada, n/a, portugal",20.000000,6.0,7.333333,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,242,"neuffen, baden-wuerttemberg, germany",37.000000,33.0,8.545455,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
4,243,"arden hills, minnesota, usa",35.875148,18.0,7.277778,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0


In [64]:
from brs.preprocessing.pipeline import BooksClearifyPublicationYear, BooksSetDefaultTypes
from brs.feature_engineering.pipeline import BookSetElderityFeature, BookSetPopularityFeature, BookSetRatingFeature

books_pipeline = DataframePipeline([
    BooksSetDefaultTypes(),
    NaPreprocessing({
        "Year-Of-Publication": {"strategy": "mean"},
        "Book-Author": {"strategy": "constant", "const_value": "unknown"},
        "Publisher": {"strategy": "constant", "const_value": "unknown"},
    }),
    BooksClearifyPublicationYear(),
    BookSetElderityFeature(),
    BookSetPopularityFeature(ratings_df=train, log=False),
    BookSetRatingFeature(ratings_df=train),
    NaPreprocessing({
        "Popularity": {"strategy": "mean"},
        "Mean-Rating": {"strategy": "mean"},
    })
])

In [65]:
data_books = books_pipeline.fit_transform(data_books)

In [66]:
data_books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Elderity,Elderity_log,Popularity,Mean-Rating
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,21,3.091042,1.905341,7.409485
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,22,3.135494,3.000000,7.333333
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,32,3.496508,2.000000,7.500000
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,24,3.218876,3.000000,6.666667
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,24,3.218876,1.905341,7.409485


In [67]:
print(train.shape)
train.head()

(129884, 3)


,User-ID,ISBN,Book-Rating
2,276762,0380711524,5
13,276762,3453092007,8
18,276762,3453213025,3
24,276762,N3453124715,4
25,276786,2864322102,6


In [68]:
train = train.merge(data_users, how="inner", on="User-ID").merge(data_books, how="inner", on="ISBN")

In [69]:
print(train.shape)
train.head()

(113625, 17)


,User-ID,ISBN,Book-Rating,Location,Age,Country,Age-Group,Num-Ratings,Mean-Rating_x,Book-Title,Book-Author,Year-Of-Publication,Publisher,Elderity,Elderity_log,Popularity,Mean-Rating_y
0,276762,0380711524,5,"duisburg, nordrhein-westfalen, germany",25.0,germany,18-29,4.0,5.0,See Jane Run,Joy Fielding,1992,Avon,31,3.465736,5.0,6.4
1,276762,3453092007,8,"duisburg, nordrhein-westfalen, germany",25.0,germany,18-29,4.0,5.0,Die zweite Haut.,Dean Koontz,1995,Heyne,28,3.367296,1.0,8.0
2,276786,8437606322,8,"madrid, madrid, spain",34.0,spain,30-44,4.0,6.5,Anna Karenina,Leo Tolstoy,1999,Ediciones Catedra S.A.,24,3.218876,1.0,8.0
3,276786,8478442588,6,"madrid, madrid, spain",34.0,spain,30-44,4.0,6.5,El Elogio de La Sombra,Tanazaki,1998,Siruela,25,3.258097,1.0,6.0
4,276798,0006379702,5,"mannheim, baden-wuerttemberg, germany",31.0,germany,30-44,3.0,6.0,"My \Star Trek\"" Memories""",William Shatner,1993,HarperCollins Publishers,30,3.433987,1.0,5.0


In [6]:
from brs.models.rate_prediction.baseline.model import BaselineRateModel

model = BaselineRateModel()
model.fit(train)

predictions = model.predict(test)


### Считаем качество наших предсказаний
У нас три ключевых метрики
- MAE
- MSE
- RMSE

Зафиксируем их, как качество самой простой модели

In [7]:
from brs.metrics.regression_metrics import MAE, RMSE, MSE

mae_score = MAE().estimate(predictions, test["Book-Rating"])
rmse_score = RMSE().estimate(predictions, test["Book-Rating"])
mse_score = MSE().estimate(predictions, test["Book-Rating"])

print(f"MAE: {mae_score}")
print(f"RMSE: {rmse_score}")
print(f"MSE: {mse_score}")

MAE: 1.5853240230307486
RMSE: 2.0801353624948558
MSE: 4.326963126301605
